## 1. Imports & Core Dependencies
This cell loads all Python libraries required for data ingestion, feature processing, and modelling.  
It also ensures consistent datetime and filesystem handling across the notebook.

In [11]:
#Imports 
import os, json, datetime
import numpy as np
import pandas as pd
import fastf1 as ff1

from pathlib import Path
import datetime


## 2. Output Directory Sanity Check
This cell verifies that the notebook has write access to the processed data directory.  
A temporary file is written to confirm relative paths resolve correctly.

In [12]:
#Path Checker Cell

OUT_DIR   = "./data_processed"

OUT_DIR = Path(OUT_DIR)   # <-- this is the missing line
testp_file = OUT_DIR / "_parquets_path_test.txt"
with open(testp_file, "w") as f:
    f.write(f"Parquets test written at {datetime.datetime.now().isoformat()}\n")
    f.write(f"Works from the ml notebook fool\n")


print("Test file written to:", testp_file)


Test file written to: data_processed\_parquets_path_test.txt


## 3. Load Pre-Qualifying Dataset & Inspect Coverage
This cell loads the consolidated parquet dataset and inspects:
- dataset dimensions
- number of samples per season (year)

This ensures the expected seasons are present before any splitting.

In [13]:
#Checks DF shape
from pathlib import Path
import pandas as pd


DATASET_PATH = OUT_DIR / "preq_dataset.parquet"

df = pd.read_parquet(DATASET_PATH)
print(df.shape)
print(df["year"].value_counts().sort_index())

(3456, 16)
year
2018    420
2019    420
2020    339
2021    440
2022    440
2023    438
2024    479
2025    480
Name: count, dtype: int64


## 4. Temporal Train–Test Split (2022–23 Train, 2024 Test)
The dataset is split chronologically:
- Training data: 2022–2023
- Hold-out test data: 2024

This mimics real-world deployment and prevents look-ahead bias.

In [14]:
train_df = df[df["year"].isin([2022, 2023])].copy()
test_df  = df[df["year"] == 2024].copy()
print(train_df["year"].value_counts().sort_index())
print(test_df["year"].value_counts().sort_index())

print(len(train_df), len(test_df))

year
2022    440
2023    438
Name: count, dtype: int64
year
2024    479
Name: count, dtype: int64
878 479


## 5. Feature Matrix and Target Construction
This cell:
- defines the prediction target (`quali_position`)
- removes identifiers and metadata from features
- constructs X/y for both training and test splits

Only pre-qualifying information is retained in X to avoid leakage.

In [15]:
TARGET = "quali_position"
DROP_COLS = ["year", "round", "event_name", "country", "location", "driver"]

X_train = train_df.drop(columns=DROP_COLS + [TARGET])
y_train = train_df[TARGET]

X_test = test_df.drop(columns=DROP_COLS + [TARGET])
y_test = test_df[TARGET]

X = df.drop(columns=DROP_COLS + [TARGET], errors="ignore")
y = df[TARGET].astype(float)

## 6. Baseline Model with Leave-One-Year-Out (LOYO) Validation
A baseline XGBoost regressor is evaluated using LOYO cross-validation
within the training period (2022–2023).

Each fold holds out one full season to assess temporal generalisation.

In [16]:
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor

groups = train_df["year"]

gkf = GroupKFold(n_splits=groups.nunique())

maes = []

for train_idx, val_idx in gkf.split(X_train, y_train, groups):
    Xtr, Xval = X_train.iloc[train_idx], X_train.iloc[val_idx]
    ytr, yval = y_train.iloc[train_idx], y_train.iloc[val_idx]

    model = XGBRegressor(
        n_estimators=200,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        objective="reg:squarederror"
    )

    model.fit(Xtr, ytr)
    preds = model.predict(Xval)
    maes.append(mean_absolute_error(yval, preds))

print("Baseline LOYO MAE:", sum(maes)/len(maes))

Baseline LOYO MAE: 5.200734228718424


## 7. Hyperparameter Optimisation via Optuna (LOYO Objective)
Optuna is used to tune XGBoost hyperparameters.

The optimisation objective is the mean LOYO MAE across training seasons,
ensuring hyperparameters generalise temporally rather than overfitting.

In [17]:
import optuna
import numpy as np
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor

# assumes these already exist from your baseline cell:
# X_train, y_train, train_df
# groups = train_df["year"]
# gkf = GroupKFold(n_splits=groups.nunique())

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 800),
        "max_depth": trial.suggest_int("max_depth", 2, 6),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "objective": "reg:squarederror",
        "random_state": 42,
        "n_jobs": -1,
    }

    maes = []

    for tr_idx, val_idx in gkf.split(X_train, y_train, groups):
        model = XGBRegressor(**params)
        model.fit(X_train.iloc[tr_idx], y_train.iloc[tr_idx])
        preds = model.predict(X_train.iloc[val_idx])
        maes.append(mean_absolute_error(y_train.iloc[val_idx], preds))

    return float(np.mean(maes))

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=40)

print("Best CV (LOYO) MAE:", study.best_value)
print("Best params:", study.best_params)

[I 2026-03-25 01:39:55,734] A new study created in memory with name: no-name-7297d898-d3d0-4cd5-996a-62006b95f80b
[I 2026-03-25 01:39:56,206] Trial 0 finished with value: 5.470744091529676 and parameters: {'n_estimators': 352, 'max_depth': 4, 'learning_rate': 0.05212327792880275, 'subsample': 0.8124114036225536, 'colsample_bytree': 0.6355337956573349}. Best is trial 0 with value: 5.470744091529676.
[I 2026-03-25 01:39:56,774] Trial 1 finished with value: 5.602656057397821 and parameters: {'n_estimators': 322, 'max_depth': 5, 'learning_rate': 0.12483578382865525, 'subsample': 0.6528112164932922, 'colsample_bytree': 0.7847374539992994}. Best is trial 0 with value: 5.470744091529676.
[I 2026-03-25 01:39:58,012] Trial 2 finished with value: 5.474103369551756 and parameters: {'n_estimators': 592, 'max_depth': 6, 'learning_rate': 0.024155310643745452, 'subsample': 0.9740725090213224, 'colsample_bytree': 0.6433902993084251}. Best is trial 0 with value: 5.470744091529676.
[I 2026-03-25 01:39:5

Best CV (LOYO) MAE: 5.019618614415699
Best params: {'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.01019633687364971, 'subsample': 0.8957180705240103, 'colsample_bytree': 0.951552092547248}


## 8. LOYO Error Breakdown by Season
This cell evaluates the tuned model using LOYO validation
and reports MAE separately for each held-out year.

This helps diagnose season-specific performance degradation.

In [18]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor

def eval_loyo_mae(X, y, groups, params):
    """
    LOYO evaluation: holds out one group (year) per fold.
    Returns dict: {year: MAE}
    """
    groups = pd.Series(groups).reset_index(drop=True)
    X = X.reset_index(drop=True)
    y = y.reset_index(drop=True)

    gkf = GroupKFold(n_splits=groups.nunique())
    fold_mae = {}

    for tr_idx, te_idx in gkf.split(X, y, groups):
        yr_vals = groups.iloc[te_idx].unique()
        if len(yr_vals) != 1:
            raise ValueError(f"Expected exactly one year in test fold, got: {yr_vals}")
        yr = int(yr_vals[0])

        model = XGBRegressor(**params, objective="reg:squarederror", random_state=42, n_jobs=-1)
        model.fit(X.iloc[tr_idx], y.iloc[tr_idx])
        pred = model.predict(X.iloc[te_idx])

        fold_mae[yr] = mean_absolute_error(y.iloc[te_idx], pred)

    return fold_mae

# ---- use the right variables for your 22-23 train split ----
best_params = study.best_params

groups_train = train_df["year"]          # 2022 & 2023 only
fold_mae = eval_loyo_mae(X_train, y_train, groups_train, best_params)

print("LOYO MAE by year (train only):")
for yr in sorted(fold_mae):
    print(yr, "->", fold_mae[yr])

print("Mean LOYO MAE:", float(np.mean(list(fold_mae.values()))))

LOYO MAE by year (train only):
2022 -> 5.00340446992354
2023 -> 5.035832758907858
Mean LOYO MAE: 5.019618614415699


## 9. Final Model Training and 2024 Hold-Out Evaluation
The best-performing model is retrained on all 2022–2023 data
and evaluated once on the unseen 2024 season.

This MAE represents the true out-of-sample performance.

In [19]:
best_model = XGBRegressor(**best_params, objective="reg:squarederror", random_state=42, n_jobs=-1)
best_model.fit(X_train, y_train)
pred_24 = best_model.predict(X_test)

print("Final 2024 MAE:", mean_absolute_error(y_test, pred_24))

Final 2024 MAE: 4.842356948613623


In [20]:
import joblib
joblib.dump(best_model, "model.pkl")
joblib.dump(list(X_train.columns), "features.pkl")

['features.pkl']